In [1]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_regulatory_summary
#
# Layer
# -----
# Gold Layer - Data Products
#
# Purpose
# -------
# Build a regulatory reporting data product combining IFRS 9,
# Basel III and stress-testing measures into a curated summary.
#
# Grain
# -----
# One row per:
# scenario month × scenario × IFRS 9 stage × country × industry
#
# Output
# ------
# gold_regulatory_summary
#
# Business Consumers
# ------------------
# • Chief Risk Officer
# • Finance and Regulatory Reporting
# • Internal Audit
# • Portfolio Risk Management
# • Power BI
# • AI Decision Intelligence
#
# Enterprise Concepts
# -------------------
# ✓ IFRS 9
# ✓ Basel III
# ✓ Regulatory Reporting
# ✓ Stress Testing
# ✓ Scenario Analysis
# ✓ Gold Data Product
# ============================================================

from pyspark.sql.functions import *
from datetime import datetime

loan_fact_table = "fact_loan_exposure"
ecl_fact_table = "fact_expected_credit_loss"
stress_fact_table = "fact_stress_testing"
country_dim_table = "dim_country"
industry_dim_table = "dim_industry"

target_table = "gold_regulatory_summary"
pipeline_name = "nb_build_regulatory_summary"

reporting_currency = "EUR"
regulatory_framework = "IFRS 9 / Basel III"

run_start_time = datetime.now()

print("ERIP Regulatory Summary Build Started")

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 3, Finished, Available, Finished, False)

ERIP Regulatory Summary Build Started


In [2]:
# ============================================================
# SECTION 2 - READ GOLD FACTS AND DIMENSIONS
# ============================================================
#
# Purpose
# -------
# Load detailed Gold facts and conformed dimensions used to
# build the regulatory reporting data product.
# ============================================================

fact_loan_exposure = spark.table(loan_fact_table)
fact_expected_credit_loss = spark.table(ecl_fact_table)
fact_stress_testing = spark.table(stress_fact_table)
dim_country = spark.table(country_dim_table)
dim_industry = spark.table(industry_dim_table)

print(f"Loan Exposure Rows : {fact_loan_exposure.count()}")
print(f"ECL Rows           : {fact_expected_credit_loss.count()}")
print(f"Stress Test Rows   : {fact_stress_testing.count()}")
print(f"Country Rows       : {dim_country.count()}")
print(f"Industry Rows      : {dim_industry.count()}")

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 4, Finished, Available, Finished, False)

Loan Exposure Rows : 5000
ECL Rows           : 5000
Stress Test Rows   : 540000
Country Rows       : 6
Industry Rows      : 10


In [3]:
# ============================================================
# SECTION 3 - BUILD BASE REGULATORY EXPOSURE DATASET
# ============================================================
#
# Purpose
# -------
# Combine loan exposure, ECL, country and industry attributes
# at loan grain before regulatory aggregation.
#
# Key Measures
# ------------
# • Exposure at Default
# • Outstanding Balance
# • Approved Limit
# • Risk Weighted Assets
# • Expected Credit Loss
# • PD / LGD
# • IFRS 9 Stage
# ============================================================

base_regulatory_df = (
    fact_expected_credit_loss.alias("e")
    .join(
        fact_loan_exposure.select(
            "loan_sk",
            "loan_id",
            "customer_sk",
            "country_sk",
            "industry_sk",
            "approved_limit",
            "outstanding_balance",
            "risk_weighted_assets",
            "risk_weight"
        ).alias("l"),
        ["loan_sk", "loan_id", "customer_sk", "country_sk", "industry_sk"],
        "left"
    )
    .join(
        dim_country.select(
            "country_sk",
            "country_code",
            "country",
            "region"
        ).alias("c"),
        "country_sk",
        "left"
    )
    .join(
        dim_industry.select(
            "industry_sk",
            "industry_code",
            "industry_name",
            "nace_code"
        ).alias("i"),
        "industry_sk",
        "left"
    )
    .select(
        col("e.loan_sk"),
        col("e.loan_id"),
        col("e.customer_sk"),
        col("e.customer_id"),
        col("e.country_sk"),
        col("e.industry_sk"),
        col("c.country_code"),
        col("c.country"),
        col("c.region"),
        col("i.industry_code"),
        col("i.industry_name"),
        col("i.nace_code"),
        col("e.ifrs9_stage_numeric"),
        col("e.ifrs9_stage_recommendation"),
        col("e.current_internal_grade"),
        col("e.pd"),
        col("e.lgd"),
        col("e.exposure_at_default"),
        col("e.calculated_ecl"),
        col("l.approved_limit"),
        col("l.outstanding_balance"),
        col("l.risk_weight"),
        col("l.risk_weighted_assets")
    )
)

print(f"Base regulatory rows: {base_regulatory_df.count()}")
display(base_regulatory_df.limit(10))

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 5, Finished, Available, Finished, False)

Base regulatory rows: 5000


SynapseWidget(Synapse.DataFrame, ed1f4899-7352-4ab1-b9ad-6a0e95cbeb1f)

In [4]:
# ============================================================
# SECTION 4 - AGGREGATE BASE REGULATORY METRICS
# ============================================================
#
# Purpose
# -------
# Aggregate IFRS 9 and Basel III measures by stage, country
# and industry before adding monthly stress scenarios.
#
# Grain
# -----
# IFRS 9 stage × country × industry
# ============================================================

base_regulatory_summary = (
    base_regulatory_df
    .groupBy(
        "ifrs9_stage_numeric",
        "ifrs9_stage_recommendation",
        "country_code",
        "country",
        "region",
        "industry_code",
        "industry_name",
        "nace_code"
    )
    .agg(
        countDistinct("loan_id").alias("loan_count"),
        countDistinct("customer_id").alias("customer_count"),
        sum("approved_limit").alias("total_approved_limit"),
        sum("outstanding_balance").alias("total_outstanding_balance"),
        sum("exposure_at_default").alias("total_ead"),
        sum("calculated_ecl").alias("base_ecl"),
        sum("risk_weighted_assets").alias("total_rwa"),
        avg("risk_weight").alias("average_risk_weight"),
        avg("pd").alias("average_pd"),
        avg("lgd").alias("average_lgd")
    )
    .withColumn(
        "minimum_capital_requirement",
        col("total_rwa") * lit(0.08)
    )
)

print(f"Base regulatory summary rows: {base_regulatory_summary.count()}")
display(base_regulatory_summary.limit(10))

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 6, Finished, Available, Finished, False)

Base regulatory summary rows: 128


SynapseWidget(Synapse.DataFrame, ef5481a6-d1c1-4c2d-9634-b575c2a6db42)

In [5]:
# ============================================================
# SECTION 5 - AGGREGATE MONTHLY STRESS TEST RESULTS
# ============================================================
#
# Purpose
# -------
# Aggregate loan-level monthly stress-testing results by
# scenario, month, IFRS stage, country and industry.
#
# Measures
# --------
# • Stressed ECL
# • ECL Increase
# • Capital Impact
# • Stressed PD
# • Stressed LGD
# ============================================================

stress_summary = (
    fact_stress_testing.alias("s")
    .join(
        fact_expected_credit_loss.select(
            "loan_sk",
            "ifrs9_stage_numeric"
        ).alias("e"),
        "loan_sk",
        "left"
    )
    .join(
        dim_country.select(
            "country_sk",
            "country_code",
            "country",
            "region"
        ).alias("c"),
        "country_sk",
        "left"
    )
    .join(
        dim_industry.select(
            "industry_sk",
            "industry_code",
            "industry_name",
            "nace_code"
        ).alias("i"),
        "industry_sk",
        "left"
    )
    .groupBy(
        col("s.scenario_sk"),
        col("s.scenario_id"),
        col("s.scenario_name"),
        col("s.scenario_rank"),
        col("s.scenario_severity"),
        col("s.scenario_month"),
        col("s.stress_intensity"),
        col("e.ifrs9_stage_numeric"),
        col("c.country_code"),
        col("c.country"),
        col("c.region"),
        col("i.industry_code"),
        col("i.industry_name"),
        col("i.nace_code")
    )
    .agg(
        countDistinct("s.loan_id").alias("stress_loan_count"),
        countDistinct("s.customer_id").alias("stress_customer_count"),
        sum("s.exposure_at_default").alias("stress_total_ead"),
        sum("s.base_ecl").alias("stress_base_ecl"),
        sum("s.stressed_ecl").alias("stressed_ecl"),
        sum("s.ecl_increase_amount").alias("ecl_increase_amount"),
        avg("s.ecl_increase_pct").alias("average_ecl_increase_pct"),
        sum("s.capital_impact_estimate").alias("capital_impact_estimate"),
        avg("s.stressed_pd").alias("average_stressed_pd"),
        avg("s.stressed_lgd").alias("average_stressed_lgd")
    )
)

print(f"Stress summary rows: {stress_summary.count()}")
display(stress_summary.limit(10))

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 7, Finished, Available, Finished, False)

Stress summary rows: 13824


SynapseWidget(Synapse.DataFrame, ffa89c33-4cdf-43c1-8326-dc040d3d44c9)

In [6]:
# ============================================================
# SECTION 6 - BUILD REGULATORY SUMMARY DATA PRODUCT
# ============================================================
#
# Purpose
# -------
# Combine monthly stress-testing results with base regulatory
# measures to create a reporting-ready Gold data product.
#
# Grain
# -----
# Scenario month × scenario × IFRS stage × country × industry
# ============================================================

gold_regulatory_summary = (
    stress_summary.alias("s")
    .join(
        base_regulatory_summary.alias("b"),
        (
            (col("s.ifrs9_stage_numeric") == col("b.ifrs9_stage_numeric")) &
            (col("s.country_code") == col("b.country_code")) &
            (col("s.industry_code") == col("b.industry_code"))
        ),
        "left"
    )
    .withColumn(
        "regulatory_risk_level",
        when(col("s.average_stressed_pd") >= 0.20, "Critical")
        .when(col("s.average_stressed_pd") >= 0.10, "High")
        .when(col("s.average_stressed_pd") >= 0.05, "Medium")
        .otherwise("Low")
    )
    .withColumn(
        "capital_adequacy_pressure",
        when(
            col("s.capital_impact_estimate") >=
            col("b.minimum_capital_requirement") * lit(0.25),
            "Severe"
        )
        .when(
            col("s.capital_impact_estimate") >=
            col("b.minimum_capital_requirement") * lit(0.10),
            "Elevated"
        )
        .otherwise("Normal")
    )
    .select(
        col("s.scenario_sk"),
        col("s.scenario_id"),
        col("s.scenario_name"),
        col("s.scenario_rank"),
        col("s.scenario_severity"),
        col("s.scenario_month"),
        date_format(col("s.scenario_month"), "yyyy-MM").alias("reporting_period"),
        col("s.stress_intensity"),
        col("s.ifrs9_stage_numeric"),
        concat(lit("Stage "), col("s.ifrs9_stage_numeric")).alias("ifrs9_stage"),
        col("s.country_code"),
        col("s.country"),
        col("s.region"),
        col("s.industry_code"),
        col("s.industry_name"),
        col("s.nace_code"),
        col("b.loan_count"),
        col("b.customer_count"),
        col("b.total_approved_limit"),
        col("b.total_outstanding_balance"),
        col("b.total_ead"),
        col("b.base_ecl"),
        col("b.total_rwa"),
        col("b.average_risk_weight"),
        col("b.average_pd"),
        col("b.average_lgd"),
        col("b.minimum_capital_requirement"),
        col("s.stressed_ecl"),
        col("s.ecl_increase_amount"),
        col("s.average_ecl_increase_pct"),
        col("s.capital_impact_estimate"),
        col("s.average_stressed_pd"),
        col("s.average_stressed_lgd"),
        col("regulatory_risk_level"),
        col("capital_adequacy_pressure"),
        lit(regulatory_framework).alias("regulatory_framework"),
        lit(reporting_currency).alias("reporting_currency"),
        current_timestamp().alias("gold_updated_timestamp")
    )
)

print(f"Regulatory summary rows created: {gold_regulatory_summary.count()}")
display(gold_regulatory_summary.limit(10))

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 8, Finished, Available, Finished, False)

Regulatory summary rows created: 13824


SynapseWidget(Synapse.DataFrame, 6a5e9c4b-4451-4103-b770-79e07bf2b4e5)

In [7]:
# ============================================================
# SECTION 7 - REGULATORY SUMMARY QUALITY VALIDATION
# ============================================================
#
# Validation Checks
# -----------------
# • Required scenario keys populated
# • IFRS 9 stage populated
# • Country and industry populated
# • No negative exposure, ECL or RWA
# • Stressed ECL not below zero
# ============================================================

total_rows = gold_regulatory_summary.count()

null_scenario_sk = gold_regulatory_summary.filter(
    col("scenario_sk").isNull()
).count()

null_ifrs_stage = gold_regulatory_summary.filter(
    col("ifrs9_stage_numeric").isNull()
).count()

null_country = gold_regulatory_summary.filter(
    col("country_code").isNull()
).count()

null_industry = gold_regulatory_summary.filter(
    col("industry_code").isNull()
).count()

negative_ead = gold_regulatory_summary.filter(
    col("total_ead") < 0
).count()

negative_base_ecl = gold_regulatory_summary.filter(
    col("base_ecl") < 0
).count()

negative_stressed_ecl = gold_regulatory_summary.filter(
    col("stressed_ecl") < 0
).count()

negative_rwa = gold_regulatory_summary.filter(
    col("total_rwa") < 0
).count()

print("Regulatory Summary Quality Checks")
print("---------------------------------")
print(f"Rows                 : {total_rows}")
print(f"Null Scenario SK     : {null_scenario_sk}")
print(f"Null IFRS Stage      : {null_ifrs_stage}")
print(f"Null Country         : {null_country}")
print(f"Null Industry        : {null_industry}")
print(f"Negative EAD         : {negative_ead}")
print(f"Negative Base ECL    : {negative_base_ecl}")
print(f"Negative Stressed ECL: {negative_stressed_ecl}")
print(f"Negative RWA         : {negative_rwa}")

if (
    null_scenario_sk > 0 or
    null_ifrs_stage > 0 or
    null_country > 0 or
    null_industry > 0 or
    negative_ead > 0 or
    negative_base_ecl > 0 or
    negative_stressed_ecl > 0 or
    negative_rwa > 0
):
    raise Exception("Regulatory Summary Validation Failed")
else:
    print("✓ Regulatory Summary Validation Passed")

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 9, Finished, Available, Finished, False)

Regulatory Summary Quality Checks
---------------------------------
Rows                 : 13824
Null Scenario SK     : 0
Null IFRS Stage      : 0
Null Country         : 0
Null Industry        : 0
Negative EAD         : 0
Negative Base ECL    : 0
Negative Stressed ECL: 0
Negative RWA         : 0
✓ Regulatory Summary Validation Passed


In [8]:
# ============================================================
# SECTION 8 - WRITE GOLD REGULATORY SUMMARY
# ============================================================

(
    gold_regulatory_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(target_table)
)

print(f"✓ Gold data product created: {target_table}")
print(f"Rows written: {gold_regulatory_summary.count()}")

StatementMeta(, 2e900d4e-4532-4c9d-a455-a78e0d1cb3d1, 10, Finished, Available, Finished, False)

✓ Gold data product created: gold_regulatory_summary
Rows written: 13824
